In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from collections import deque

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

In [2]:
WINDOW_SIZE = 30
WEBCAM_ID   = 0

# Exercise label decoder (must match encoding used in 04_tabular_data_creation.ipynb)
EXERCISE_LABELS = {
    0: "Pull-up",
    1: "Push-up",
    2: "Russian Twist",
    3: "Squat",
}

MODEL_PATH           = Path("../data/models/pose_landmarker_lite.task")
SCALER_PATH          = Path("../data/models/scaler.pkl")
EXERCISE_SVM_PATH    = Path("../data/models/svm_exercise_tuned.pkl")
CORRECTNESS_SVM_PATH = Path("../data/models/svm_correctness_tuned.pkl")
EXERCISE_RF_PATH     = Path("../data/models/rf_exercise_tuned.pkl")
CORRECTNESS_RF_PATH  = Path("../data/models/rf_correctness_tuned.pkl")

for p in [MODEL_PATH, SCALER_PATH, EXERCISE_SVM_PATH,
          CORRECTNESS_SVM_PATH, EXERCISE_RF_PATH, CORRECTNESS_RF_PATH]:
    assert p.exists(), f"❌ File not found: {p}"

print("✅ Config ready.")
print(f"   Rolling window : {WINDOW_SIZE} frames")
print(f"   Webcam ID      : {WEBCAM_ID}")
print(f"   Exercise labels: {EXERCISE_LABELS}")

✅ Config ready.
   Rolling window : 30 frames
   Webcam ID      : 0
   Exercise labels: {0: 'Pull-up', 1: 'Push-up', 2: 'Russian Twist', 3: 'Squat'}


In [3]:
def calculate_angle(a, b, c):
    """Angle at B formed by A->B->C, in degrees [0, 180]."""
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = (np.arctan2(c[1]-b[1], c[0]-b[0]) -
               np.arctan2(a[1]-b[1], a[0]-b[0]))
    angle = np.abs(np.degrees(radians))
    if angle > 180:
        angle = 360 - angle
    return round(angle, 2)

print(f"✅ calculate_angle ready. Sanity check (expect 90°): {calculate_angle((0,0),(1,0),(1,1))}°")

✅ calculate_angle ready. Sanity check (expect 90°): 90.0°


In [4]:
LANDMARKS = {
    "left_shoulder": 11, "right_shoulder": 12,
    "left_elbow"   : 13, "right_elbow"   : 14,
    "left_wrist"   : 15, "right_wrist"   : 16,
    "left_hip"     : 23, "right_hip"     : 24,
    "left_knee"    : 25, "right_knee"    : 26,
    "left_ankle"   : 27, "right_ankle"   : 28,
    "left_heel"    : 29, "right_heel"    : 30,
}

def get_coords(landmarks, name):
    lm = landmarks[LANDMARKS[name]]
    return (lm.x, lm.y)

def get_z(landmarks, name):
    return landmarks[LANDMARKS[name]].z

def extract_angles(landmarks):
    """13 features: 10 joint angles + 3 torso rotation (matches training data)."""
    a = {}

    a["left_elbow_angle"]    = calculate_angle(get_coords(landmarks,"left_shoulder"),  get_coords(landmarks,"left_elbow"),   get_coords(landmarks,"left_wrist"))
    a["right_elbow_angle"]   = calculate_angle(get_coords(landmarks,"right_shoulder"), get_coords(landmarks,"right_elbow"),  get_coords(landmarks,"right_wrist"))
    a["left_shoulder_angle"] = calculate_angle(get_coords(landmarks,"left_elbow"),     get_coords(landmarks,"left_shoulder"),get_coords(landmarks,"left_hip"))
    a["right_shoulder_angle"]= calculate_angle(get_coords(landmarks,"right_elbow"),    get_coords(landmarks,"right_shoulder"),get_coords(landmarks,"right_hip"))
    a["left_hip_angle"]      = calculate_angle(get_coords(landmarks,"left_shoulder"),  get_coords(landmarks,"left_hip"),     get_coords(landmarks,"left_knee"))
    a["right_hip_angle"]     = calculate_angle(get_coords(landmarks,"right_shoulder"), get_coords(landmarks,"right_hip"),    get_coords(landmarks,"right_knee"))
    a["left_knee_angle"]     = calculate_angle(get_coords(landmarks,"left_hip"),       get_coords(landmarks,"left_knee"),    get_coords(landmarks,"left_ankle"))
    a["right_knee_angle"]    = calculate_angle(get_coords(landmarks,"right_hip"),      get_coords(landmarks,"right_knee"),   get_coords(landmarks,"right_ankle"))
    a["left_ankle_angle"]    = calculate_angle(get_coords(landmarks,"left_knee"),      get_coords(landmarks,"left_ankle"),   get_coords(landmarks,"left_heel"))
    a["right_ankle_angle"]   = calculate_angle(get_coords(landmarks,"right_knee"),     get_coords(landmarks,"right_ankle"),  get_coords(landmarks,"right_heel"))

    a["shoulder_z_diff"] = round(get_z(landmarks,"left_shoulder") - get_z(landmarks,"right_shoulder"), 4)
    a["hip_z_diff"]      = round(get_z(landmarks,"left_hip")      - get_z(landmarks,"right_hip"),      4)
    a["torso_rotation"]  = round(a["shoulder_z_diff"] - a["hip_z_diff"], 4)

    return a

print("✅ extract_angles ready. Features per frame: 13 angles + 13 velocities = 26.")

✅ extract_angles ready. Features per frame: 13 angles + 13 velocities = 26.


In [5]:
# Pose connections to draw (index pairs from MediaPipe's 33-landmark schema)
POSE_CONNECTIONS = [
    (11, 12),             # shoulders
    (11, 13), (13, 15),   # left arm
    (12, 14), (14, 16),   # right arm
    (11, 23), (12, 24),   # torso sides
    (23, 24),             # hips
    (23, 25), (25, 27), (27, 29),  # left leg
    (24, 26), (26, 28), (28, 30),  # right leg
]

def draw_skeleton(frame, landmarks, h, w):
    """
    Draw pose landmarks and connections onto a frame (in-place).

    Args:
        frame     : BGR OpenCV frame
        landmarks : list of NormalizedLandmark from MediaPipe Tasks API
        h, w      : frame height and width (for denormalization)
    """
    # Draw connections
    for start_idx, end_idx in POSE_CONNECTIONS:
        if start_idx >= len(landmarks) or end_idx >= len(landmarks):
            continue
        x1 = int(landmarks[start_idx].x * w)
        y1 = int(landmarks[start_idx].y * h)
        x2 = int(landmarks[end_idx].x * w)
        y2 = int(landmarks[end_idx].y * h)
        cv2.line(frame, (x1, y1), (x2, y2), (200, 200, 200), 2)

    # Draw landmark dots (only the ones we use)
    for name, idx in LANDMARKS.items():
        if idx >= len(landmarks):
            continue
        x = int(landmarks[idx].x * w)
        y = int(landmarks[idx].y * h)
        cv2.circle(frame, (x, y), 5, (0, 255, 0), -1)

print("✅ draw_skeleton ready.")

✅ draw_skeleton ready.


In [6]:
scaler            = joblib.load(SCALER_PATH)
exercise_svm      = joblib.load(EXERCISE_SVM_PATH)
correctness_svm   = joblib.load(CORRECTNESS_SVM_PATH)
exercise_rf       = joblib.load(EXERCISE_RF_PATH)
correctness_rf    = joblib.load(CORRECTNESS_RF_PATH)

print("✅ Scaler and models loaded.")
print(f"   Scaler expects {scaler.n_features_in_} features.")

✅ Scaler and models loaded.
   Scaler expects 26 features.


In [7]:
def majority_vote(iterable):
    """Return the most common value in a sequence."""
    from collections import Counter
    return Counter(iterable).most_common(1)[0][0]

def predict_from_window(window, prev_angles_ref, scaler, ex_svm, corr_svm, ex_rf, corr_rf):
    """
    Given a deque of (angles_dict) from the last N frames,
    scale them, run all 4 models, and return majority-voted predictions
    plus per-model correctness confidence (% of frames predicted correct).

    prev_angles_ref is a 1-element list used as a mutable reference for velocity.
    Returns: (exercise_svm, correctness_svm, exercise_rf, correctness_rf,
               confidence_svm, confidence_rf)  — or None if window is empty.
    """
    if not window:
        return None

    rows = []
    prev = None
    # Recompute velocities over the window snapshot
    for angles in window:
        if prev is not None:
            velocities = {f"{k}_velocity": round(abs(angles[k] - prev[k]) * 30, 2)
                          for k in angles}
        else:
            velocities = {f"{k}_velocity": 0.0 for k in angles}
        rows.append({**angles, **velocities})
        prev = angles

    df = pd.DataFrame(rows)

    # Velocity noise cap
    vel_cols = [c for c in df.columns if "velocity" in c]
    df[vel_cols] = df[vel_cols].clip(upper=3000)

    # Feature alignment guard
    feature_cols = [c for c in df.columns]
    assert len(feature_cols) == scaler.n_features_in_, (
        f"Feature mismatch: got {len(feature_cols)}, expected {scaler.n_features_in_}"
    )

    X_scaled = scaler.transform(df[feature_cols])

    pred_ex_svm   = majority_vote(ex_svm.predict(X_scaled))
    pred_ex_rf    = majority_vote(ex_rf.predict(X_scaled))

    corr_preds_svm = corr_svm.predict(X_scaled)
    corr_preds_rf  = corr_rf.predict(X_scaled)

    pred_corr_svm  = majority_vote(corr_preds_svm)
    pred_corr_rf   = majority_vote(corr_preds_rf)

    conf_svm = round((corr_preds_svm == 1).sum() / len(corr_preds_svm) * 100, 1)
    conf_rf  = round((corr_preds_rf  == 1).sum() / len(corr_preds_rf)  * 100, 1)

    return pred_ex_svm, pred_corr_svm, pred_ex_rf, pred_corr_rf, conf_svm, conf_rf

print("✅ Inference helpers ready.")

✅ Inference helpers ready.


In [8]:
import warnings
warnings.filterwarnings("ignore")

# --- Overlay drawing helper ---
def draw_overlay(frame, pred_ex_svm, pred_corr_svm, pred_ex_rf,
                 pred_corr_rf, conf_svm, conf_rf, window_fill):
    h, w = frame.shape[:2]

    label_map = {1: "Correct",   0: "Incorrect"}
    color_map = {1: (0, 220, 0), 0: (0, 0, 220)}

    ex_svm_name = EXERCISE_LABELS.get(pred_ex_svm, str(pred_ex_svm))
    ex_rf_name  = EXERCISE_LABELS.get(pred_ex_rf,  str(pred_ex_rf))

    lines = [
        (f"Exercise (SVM): {ex_svm_name}",                        (255, 255, 255)),
        (f"Exercise  (RF): {ex_rf_name}",                         (255, 255, 255)),
        (f"Form     (SVM): {label_map.get(pred_corr_svm, '?')}",  color_map.get(pred_corr_svm, (200, 200, 200))),
        (f"Form      (RF): {label_map.get(pred_corr_rf,  '?')}",  color_map.get(pred_corr_rf,  (200, 200, 200))),
        (f"Confidence SVM: {conf_svm}% correct",                  (200, 200, 50)),
        (f"Confidence  RF: {conf_rf}% correct",                   (200, 200, 50)),
        (f"Window: {window_fill}/{WINDOW_SIZE} frames",           (150, 150, 150)),
    ]

    box_h = len(lines) * 28 + 16
    overlay = frame.copy()
    cv2.rectangle(overlay, (8, 8), (360, box_h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.5, frame, 0.5, 0, frame)

    for i, (text, color) in enumerate(lines):
        cv2.putText(frame, text, (14, 32 + i * 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.62, color, 2, cv2.LINE_AA)

    cv2.putText(frame, "Press Q in this window, or Interrupt kernel to quit",
                (14, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150, 150, 150), 1, cv2.LINE_AA)


# --- Main loop ---
base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=RunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture(WEBCAM_ID)
if not cap.isOpened():
    raise RuntimeError(f"❌ Could not open webcam (ID={WEBCAM_ID}).")

window      = deque(maxlen=WINDOW_SIZE)
prev_angles = None
last_result = None

WINDOW_NAME = "AI Personal Trainer — Live"
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

print("🎥 Webcam inference started.")
print("   Stop options: press Q in the video window, close the window, or Interrupt the kernel (■).")

try:
    with PoseLandmarker.create_from_options(options) as landmarker:
        while True:

            # --- Stop if window was closed with the X button ---
            if cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
                print("⏹️  Window closed.")
                break

            ret, frame = cap.read()
            if not ret:
                print("⚠️  Frame capture failed.")
                break

            h, w = frame.shape[:2]

            # --- Pose detection ---
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            result    = landmarker.detect(mp_image)

            if result.pose_landmarks and len(result.pose_landmarks) > 0:
                landmarks = result.pose_landmarks[0]
                draw_skeleton(frame, landmarks, h, w)

                angles = extract_angles(landmarks)
                angle_vals = {k: v for k, v in angles.items()
                              if "z_diff" not in k and "rotation" not in k}
                if all(0 <= v <= 180 for v in angle_vals.values()):
                    window.append(angles)

            if len(window) == WINDOW_SIZE:
                last_result = predict_from_window(
                    window, prev_angles,
                    scaler, exercise_svm, correctness_svm,
                    exercise_rf, correctness_rf
                )

            if last_result is not None:
                draw_overlay(frame, *last_result, window_fill=len(window))
            else:
                cv2.putText(frame, f"Warming up... ({len(window)}/{WINDOW_SIZE} frames)",
                            (14, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 50), 2)

            cv2.imshow(WINDOW_NAME, frame)

            # --- Q key to quit (works when the OpenCV window has focus) ---
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("⏹️  Stopped by Q key.")
                break

except KeyboardInterrupt:
    # Triggered by the Jupyter Interrupt kernel button (■)
    print("⏹️  Interrupted by kernel.")

finally:
    # Always runs — guarantees webcam and windows are released
    cap.release()
    cv2.destroyAllWindows()
    # Extra waitKey calls flush any pending GUI events on some systems
    for _ in range(5):
        cv2.waitKey(1)
    print("✅ Webcam released and windows closed.")

🎥 Webcam inference started.
   Stop options: press Q in the video window, close the window, or Interrupt the kernel (■).
⏹️  Window closed.
✅ Webcam released and windows closed.
